# Notebook 12: Error-Driven Augmentation

## Purpose
Evaluate classifier-specific error-driven augmentation — the most principled
augmentation strategy in this project. Unlike Notebooks 10-11 which used a
single shared synthetic pool, each classifier here receives augmentation
tailored to its own failure modes.

## Methodology
For each classifier (LR, MLP-1, MLP-2):
1. False positives on the training set were identified at the Notebook 09
   threshold (τ=0.69 for all three)
2. K-means clustering (k=200) applied to false positive embeddings —
   selects diverse, representative failure modes rather than redundant
   near-duplicate errors
3. Sentence closest to each centroid selected as generation seed
4. `Llama-3-8B-Instruct` (temperature=0.7) rewrites each seed as a
   genuine hedge — preserving topic and vocabulary but adding real
   epistemic uncertainty
5. Two variants generated per seed, each using a different hedging device
6. Filtered by cosine similarity to nearest real positive (τ=0.8) —
   same filter as Notebook 10

## Why Classifier-Specific?
Each model makes structurally different errors:
- LR: 5,083 false positives (7.3%) — over-predicts due to linear boundary
- MLP-1: 1,134 false positives (1.6%) — more selective, harder failure modes
- MLP-2: 1,473 false positives (2.1%) — intermediate error profile

Augmenting with a shared pool would mix error types across models.
Classifier-specific augmentation ensures each model is challenged on
its own blind spots.

## Generated Files
| Classifier | Seeds | Raw variants | Source |
|---|---|---|---|
| LR | 199 | 398 | `error_driven_lr_raw.parquet` |
| MLP-1 | 200 | ~400 | `error_driven_mlp1_raw.parquet` |
| MLP-2 | 200 | ~400 | `error_driven_mlp2_raw.parquet` |

## Filtering
Cosine similarity to nearest real positive at τ=0.8 — identical to
Notebook 10. Error-driven variants are genuine hedges (label=1) so
the same filter logic applies: retain candidates geometrically close
to the real positive manifold.

## Evaluation
- UMAP: one plot per classifier condition, filtered synthetics overlaid
- Metrics table: precision, recall, F1 vs Notebooks 09-11
- DET curves: all conditions overlaid
- Key question: does classifier-specific augmentation outperform the
  shared augmentation strategies from Notebooks 10-11?

In [1]:
# -----------------------------------------------
# Imports — identical to Notebooks 09-11
# -----------------------------------------------
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import umap.umap_ as umap

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW

from sklearn.linear_model import LogisticRegression
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import (
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    det_curve,
)
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
os.chdir("..")

c:\Users\vkamat01\hedging-txtclf-experiments\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# -----------------------------------------------
# Imports — identical to Notebooks 09-11
# -----------------------------------------------
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import umap.umap_ as umap

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW

from sklearn.linear_model import LogisticRegression
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import (
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    det_curve,
)
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# -----------------------------------------------
# Paths — synthetic parquets per classifier
# These will be populated as generation completes:
#   LR:    already done
#   MLP-1: in progress
#   MLP-2: tomorrow
# -----------------------------------------------
SYNTHETIC_PATHS = {
    'LR':    "data/synthetic/error_driven_lr_raw.parquet",
    'MLP-1': "data/synthetic/error_driven_mlp1_raw.parquet",
    'MLP-2': "data/synthetic/error_driven_mlp2_raw.parquet",
}

# Real embeddings and splits
X_train = np.load("data/processed/embeddings/X_train.npy")
y_train = np.load("data/processed/embeddings/y_train.npy")
X_cal   = np.load("data/processed/embeddings/X_cal.npy")
y_cal   = np.load("data/processed/embeddings/y_cal.npy")
X_test  = np.load("data/processed/embeddings/X_test.npy")
y_test  = np.load("data/processed/embeddings/y_test.npy")

print(f"Train: {X_train.shape} | Positives: {y_train.sum()}")
print(f"Cal:   {X_cal.shape}   | Positives: {y_cal.sum()}")
print(f"Test:  {X_test.shape}  | Positives: {y_test.sum()}")

# Load prior baseline metrics for comparison
with open("data/results/metrics_09_baseline.json") as f:
    baseline_09 = json.load(f)
with open("data/results/metrics_10_positive_aug.json") as f:
    baseline_10 = json.load(f)
with open("data/results/metrics_11_contrastive_aug.json") as f:
    baseline_11 = json.load(f)

print("\nPrior baselines loaded: Notebooks 09, 10, 11")

# -----------------------------------------------
# Verify which synthetic files are ready
# -----------------------------------------------
print("\nSynthetic file status:")
for clf_name, path in SYNTHETIC_PATHS.items():
    if os.path.exists(path):
        df = pd.read_parquet(path)
        print(f"  ✓ {clf_name}: {len(df)} variants — {path}")
    else:
        print(f"  ✗ {clf_name}: not yet generated — {path}")